# Portfolio Analysis Notebook

This notebook provides comprehensive analysis of your investment portfolio including:
- Portfolio performance metrics
- Risk analysis
- Asset allocation
- Individual holding performance

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import requests
import json

# API Configuration
API_BASE_URL = 'http://localhost:5000/api'

# Portfolio ID to analyze
PORTFOLIO_ID = 1  # Change this to your portfolio ID

## 1. Portfolio Overview

In [ ]:
# Fetch portfolio performance summary
response = requests.get(f'{API_BASE_URL}/portfolio/{PORTFOLIO_ID}/performance-summary')
summary = response.json()

print(f"Portfolio: {summary.get('portfolio_name')}\n")
print(f"Total Value: ${summary.get('total_value', 0):,.2f}")
print(f"Invested Amount: ${summary.get('invested_amount', 0):,.2f}")
print(f"Total Return: ${summary.get('total_return', 0):,.2f}")
print(f"Total Return %: {summary.get('total_return_percent', 0):.2f}%")
print(f"Number of Holdings: {summary.get('holdings_count', 0)}")

## 2. Asset Allocation

In [ ]:
# Get allocation by symbol
response = requests.get(f'{API_BASE_URL}/portfolio/{PORTFOLIO_ID}/allocation')
allocation_data = response.json()['allocation']

symbols = list(allocation_data.keys())
values = [allocation_data[s]['value'] for s in symbols]

fig = px.pie(values=values, names=symbols, title='Portfolio Allocation')
fig.show()

## 3. Holdings Performance

In [ ]:
# Get holdings performance
response = requests.get(f'{API_BASE_URL}/portfolio/{PORTFOLIO_ID}/performance-summary')
holdings = response.json()['holdings_performance']

df = pd.DataFrame(holdings)
df = df[['symbol', 'quantity', 'average_cost', 'current_price', 'gain_loss', 'gain_loss_percent']]

print("Holdings Performance:")
print(df.to_string(index=False))

In [ ]:
# Visualize gains/losses
fig = px.bar(df, x='symbol', y='gain_loss_percent', 
             title='Holdings Gain/Loss %',
             color='gain_loss_percent',
             color_continuous_scale=['red', 'green'])
fig.show()

## 4. Risk Metrics

In [ ]:
# Get portfolio risk metrics
response = requests.get(f'{API_BASE_URL}/risk/portfolio-metrics/{PORTFOLIO_ID}')
risk_metrics = response.json()

print("Portfolio Risk Metrics:")
print(f"Volatility (Annual): {risk_metrics.get('volatility', 0):.4f} ({risk_metrics.get('volatility', 0)*100:.2f}%)")
print(f"Sharpe Ratio: {risk_metrics.get('sharpe_ratio', 0):.4f}")
print(f"Sortino Ratio: {risk_metrics.get('sortino_ratio', 0):.4f}")
print(f"Max Drawdown: {risk_metrics.get('max_drawdown', 0):.4f} ({risk_metrics.get('max_drawdown', 0)*100:.2f}%)")
print(f"Value at Risk (95%): {risk_metrics.get('value_at_risk_95', 0):.4f}")
print(f"Value at Risk (99%): {risk_metrics.get('value_at_risk_99', 0):.4f}")

## 5. Top Gainers and Losers

In [ ]:
# Get top gainers
response = requests.get(f'{API_BASE_URL}/portfolio/{PORTFOLIO_ID}/top-gainers?limit=5')
gainers = response.json()['top_gainers']

print("Top 5 Gainers:")
for g in gainers:
    print(f"  {g['symbol']}: {g['gain_loss_percent']:.2f}%")

In [ ]:
# Get top losers
response = requests.get(f'{API_BASE_URL}/portfolio/{PORTFOLIO_ID}/top-losers?limit=5')
losers = response.json()['top_losers']

print("Top 5 Losers:")
for l in losers:
    print(f"  {l['symbol']}: {l['gain_loss_percent']:.2f}%")